In [1]:
!pip install torch transformers datasets rouge-score bert-score evaluate nltk

  Using cached matplotlib-3.10.6-cp311-cp311-macosx_11_0_arm64.whl.metadata (11 kB)
  Using cached contourpy-1.3.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.59.2-cp311-cp311-macosx_10_9_universal2.whl.metadata (109 kB)
  Using cached kiwisolver-1.4.9-cp311-cp311-macosx_11_0_arm64.whl.metadata (6.3 kB)
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
Using cached matplotlib-3.10.6-cp311-cp311-macosx_11_0_arm64.whl (8.1 MB)
Using cached contourpy-1.3.3-cp311-cp311-macosx_11_0_arm64.whl (270 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.59.2-cp311-cp311-macosx_10_9_universal2.whl (2.8 MB)
Using cached kiwisolver-1.4.9-cp311-cp311-macosx_11_0_arm64.whl (65 kB)
Using cached pyparsing-3.2.3-py3-none-any.whl (111 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [bert-score]7 [matplotlib]


In [3]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    pipeline
)
import evaluate
import nltk
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('stopwords')

/Users/tanishapriya/Desktop/Capstone/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/tanishapriya/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/tanishapriya/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
# Bio_ClinicalBERT as encoder
biobert_model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
biobert_tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

# T5-small as summarizer
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
summarizer_tokenizer = AutoTokenizer.from_pretrained("t5-small")


In [5]:
def biobert_encode(text, tokenizer, model, max_len=512):
    """Encode clinical note into embeddings using BioBERT"""
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, padding="max_length", max_length=max_len
    )
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state  # contextual embeddings


def generate_summary(text, tokenizer, model, max_len=512, summary_len=100):
    """Generate abstractive summary using T5"""
    inputs = tokenizer(
        "summarize: " + text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=max_len,
    )
    summary_ids = model.generate(
        inputs["input_ids"],
        num_beams=4,
        length_penalty=2.0,
        max_length=summary_len,
        min_length=30,
        early_stopping=True,
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)



In [6]:
# Step 5: Example Input
# ============================
clinical_note = """
Patient is a 65-year-old male admitted after a fall at home. Presented with a right hip fracture 
and mild head trauma. Underwent successful open reduction and internal fixation of the hip. 
Post-op vitals stable, pain controlled with acetaminophen. No neurological deficits observed. 
Physical therapy initiated on day 2 post-op. Patient discharged in stable condition with 
recommendations for follow-up and continued physiotherapy.
"""

summary = generate_summary(clinical_note, summarizer_tokenizer, summarizer_model)
print("Original Note:\n", clinical_note)
print("\n Generated Summary:\n", summary)


Original Note:
 
Patient is a 65-year-old male admitted after a fall at home. Presented with a right hip fracture 
and mild head trauma. Underwent successful open reduction and internal fixation of the hip. 
Post-op vitals stable, pain controlled with acetaminophen. No neurological deficits observed. 
Physical therapy initiated on day 2 post-op. Patient discharged in stable condition with 
recommendations for follow-up and continued physiotherapy.


 Generated Summary:
 patient is a 65-year-old male admitted after a fall at home. Presented with a right hip fracture and mild head trauma. underwent successful open reduction and internal fixation of the hip.


In [7]:
# Step 6: Evaluation
# ============================

# Load metrics
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

reference_summary = """65-year-old male with hip fracture and mild head trauma. 
Successful surgery, stable recovery, started physiotherapy, discharged with follow-up advice."""

# ROUGE Scores
rouge_results = rouge.compute(
    predictions=[summary], references=[reference_summary]
)
print("\n📊 ROUGE Results:", rouge_results)

# BERTScore
bert_results = bertscore.compute(
    predictions=[summary], references=[reference_summary], lang="en"
)
print("\n📊 BERTScore Results:", bert_results)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



📊 ROUGE Results: {'rouge1': np.float64(0.43636363636363634), 'rouge2': np.float64(0.3018867924528302), 'rougeL': np.float64(0.43636363636363634), 'rougeLsum': np.float64(0.43636363636363634)}
  2025-09-07T07:06:06.122924Z  WARN  Reqwest(reqwest::Error { kind: Request, url: "https://transfer.xethub.hf.co/xorbs/default/a8b08e1910d1a72a359e8c5b54e716dd61abde0fb9619e0e621eeb469feaef52?X-Xet-Signed-Range=bytes%3D0-44875486&Expires=1757231513&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly90cmFuc2Zlci54ZXRodWIuaGYuY28veG9yYnMvZGVmYXVsdC9hOGIwOGUxOTEwZDFhNzJhMzU5ZThjNWI1NGU3MTZkZDYxYWJkZTBmYjk2MTllMGU2MjFlZWI0NjlmZWFlZjUyP1gtWGV0LVNpZ25lZC1SYW5nZT1ieXRlcyUzRDAtNDQ4NzU0ODYiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3NTcyMzE1MTN9fX1dfQ__&Signature=tRPEoUgM4llOs9VJu9bUaj216R~z~1kPszoTNbQYYBgG0ysSYfRpI0Gx8hhwUfZlGYr-p5FQ4cDEpjIDQ0GzqMnDDbVw4EQc0gfQ-kTsf8WbNO5I5WXg~tnccbIqPC9e1JEshDlCLoGJhm7FLiPBtQ72pVnj181JIkwOhynba7wHTSpaNpmjNCcjyMdetSOQKhGaAFicsHiRiFcECRDQr1NX3iVKUlFX

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



📊 BERTScore Results: {'precision': [0.892349123954773], 'recall': [0.8947582244873047], 'f1': [0.8935520052909851], 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=4.56.1)'}


In [ ]:
#Step 7: Readability
# ============================
from nltk.tokenize import sent_tokenize, word_tokenize

def flesch_reading_ease(text):
    sentences = sent_tokenize(text)
    words = word_tokenize(text)
    syllables = sum([len([c for c in word if c in "aeiouAEIOU"]) for word in words])
    ASL = len(words) / len(sentences)  # Average sentence length
    ASW = syllables / len(words)       # Avg syllables per word
    FRE = 206.835 - (1.015 * ASL) - (84.6 * ASW)
    return FRE

readability = flesch_reading_ease(summary)
print("\n📊 Readability (Flesch Score):", readability)